# Interactive Graphs with Plotly 📊

## What you will learn in this course 🧐🧐

Plotly is a powerful library for creating interactive visualizations in Python. In this course, we'll focus on the `graph_objects` API, which gives you fine-grained control over every aspect of your figures.

By the end you'll be able to:
- Build figures with the `go.Figure(data=..., layout=...)` pattern
- Compose charts with `add_trace` and `update_layout`
- Create subplots with `make_subplots`
- Add the most useful interactive elements
- Plot data on world and country maps

## The model for building a Plotly figure

When you call a Seaborn function, it creates a Matplotlib figure and axes behind the scenes. You can then tweak the axes with Matplotlib commands. 

Plotly is similar, but instead of an implicit figure, you create an explicit `go.Figure` object. You build up the `data` and `layout` dictionaries, then pass them to `go.Figure(data=..., layout=...)`.

A Plotly figure has two parts:

- **`data`**: a list of *traces*, each describing one visual layer (a scatter, a histogram, a bar series...)
- **`layout`**: everything else like title, axes, legend, annotations, geography settings, etc.

Let's import the libraries and start with a simple example:

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Custom color palette used throughout the lecture
color = ["#FE0162", "#345FFF", "#0ECDD0", "#F6970A", "#0B155B"]

# Build a custom colorscale from the palette (used for choropleth maps)
custom_colorscale = [
    [0.00, color[4]],   # deep navy
    [0.25, color[1]],   # blue
    [0.50, color[2]],   # teal
    [0.75, color[3]],   # orange
    [1.00, color[0]],   # pink/red
]

# Load the Iris dataset
df = px.data.iris()
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species,species_id
0,5.1,3.5,1.4,0.2,setosa,1
1,4.9,3.0,1.4,0.2,setosa,1
2,4.7,3.2,1.3,0.2,setosa,1
3,4.6,3.1,1.5,0.2,setosa,1
4,5.0,3.6,1.4,0.2,setosa,1


## A first scatter plot

Let's load the Iris dataset and create a simple scatter plot of sepal length vs sepal width. 

In [3]:
fig = go.Figure(
    data=go.Scatter(
        x=df['sepal_width'],
        y=df['sepal_length'],
        mode='markers',
        marker=go.scatter.Marker(color=color[0])
    ),
    layout=go.Layout(
        title=go.layout.Title(text='Sepal length vs. sepal width', x=0.5)
    )
)

fig.show()

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/thierrylegros/00-JEDHA/Fullstack-AI/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
  File "/var/folders/z5/kyyhqxqd4b966377jmszys600000gn/T/ipykernel_21502/3103306869.py", line 13, in <module>
    fig.show()
    ~~~~~~~~^^
  File "/Users/thierrylegros/00-JEDHA/Fullstack-AI/.venv/lib/python3.14/site-packages/plotly/basedatatypes.py", line 3420, in show
    return pio.show(self, *args, **kwargs)
           ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/thierrylegros/00-JEDHA/Fullstack-AI/.venv/lib/python3.14/site-packages/plotly/io/_renderers.py", line 415, in show
    raise ValueError(
        "Mime type rendering requires nbformat>=4.2.0 but it is not installed"
    )
ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/thierrylegros/00-JEDHA/Ful

**Interpretation**

Each point is one iris flower. There is a moderate positive relationship between sepal width and sepal length: flowers with wider sepals tend to be slightly longer, but the spread is wide. Notice the visible gap around sepal width 2.5 cm: a small group of flowers sits clearly above (long sepals) while another sits below. This hint of clustering will become clear once we color by species in the next plot.

The key elements in this code are:
- `go.Scatter(...)` is the trace; `mode='markers'` makes it a scatter (use `'lines'` or `'lines+markers'` for line plots)
- `go.Layout(...)` holds the title; `x=0.5` centers it
- `fig.show()` renders the interactive plot. Try hovering, zooming, or double-clicking to reset.

<Note type ='tip' title='See all options'>

in Jupyter, run `go.Scatter?` or `go.Layout?` to see all available properties.

</Note>

## Adding color, axis labels, axis ranges

Each trace and layout component has its own sub-objects you can configure. The most useful ones for scatter plots are:

- `marker` (color, size, opacity, symbol)
- `xaxis` / `yaxis` (title, range)

In [ ]:
# Map each species to one color from our palette
species_colors = {1: color[0], 2: color[1], 3: color[2]}
point_colors = df['species_id'].map(species_colors)

fig = go.Figure(
    data=go.Scatter(
        x=df['sepal_width'],
        y=df['sepal_length'],
        mode='markers',
        marker=go.scatter.Marker(color=point_colors, size=8)
    ),
    layout=go.Layout(
        title=go.layout.Title(text='Iris, colored by species', x=0.5),
        xaxis=go.layout.XAxis(title='Sepal width', range=[0, 5]),
        yaxis=go.layout.YAxis(title='Sepal length', range=[0, 10])
    )
)

fig.show()

**Interpretation**

Coloring the points by species reveals the structure that was hidden in the previous plot. One species (in pink) forms a tight, distinct cluster with shorter sepals and wider relative width. The other two species (blue and teal) overlap more with each other but still occupy a different region of the plot, with longer sepals on average. This kind of color encoding turns a vague scatter into a clear classification picture, and is the main reason coloring by category is the first thing to try on a multidimensional dataset.

## Updating an existing figure

Two methods do almost everything you'll need:

- `fig.add_trace(...)` adds a new visual layer
- `fig.update_layout(...)` changes layout properties

This pattern is much more readable than recreating the figure from scratch.

In [ ]:
species_colors = {1: color[3], 2: color[1], 3: color[2]}
point_colors = df['species_id'].map(species_colors)

fig = go.Figure(
    data=go.Scatter(
        x=df['sepal_width'],
        y=df['sepal_length'],
        mode='markers',
        marker=go.scatter.Marker(color=point_colors, size=8)
    )
)

# Add a second trace on top of the scatter
fig.add_trace(go.Histogram(x=df['sepal_width'], marker_color=color[0], opacity=0.5))

# Tweak the layout
fig.update_layout(
    title=dict(text='Scatter + histogram of sepal width', x=0.5),
    showlegend=False
)

fig.show()

**Interpretation**

This figure stacks two views on the same axes: the scatter (sepal length vs. width) and a histogram of sepal width (the orange bars). The histogram's bars are tall where many flowers share that sepal width: the peak around 3.0 cm shows that most flowers cluster there. Because both traces share the x-axis, you can see at a glance which x-values are well represented by the scatter and which are sparse. In practice you would usually put the two traces in separate subplots (next section) rather than overlaying them on the same axes, since the y-scales are not comparable.

## Subplots

When two traces shouldn't share axes, use `make_subplots` to lay them out side by side.

In [ ]:
from plotly.subplots import make_subplots

species_colors = {1: color[0], 2: color[1], 3: color[2]}
point_colors = df['species_id'].map(species_colors)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Sepal length vs. width', 'Distribution of sepal width')
)

fig.add_trace(
    go.Scatter(
        x=df['sepal_width'],
        y=df['sepal_length'],
        mode='markers',
        marker=go.scatter.Marker(color=point_colors, size=8)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=df['sepal_width'], marker_color=color[3]),
    row=1, col=2
)

fig.update_layout(showlegend=False, title=dict(text='Iris dashboard', x=0.5))
fig.show()

**Interpretation**

The same two traces are now in their own panels, with their own axes. The left panel shows the species clusters cleanly without any orange bars in the way; the right panel shows a clear histogram of sepal width with the count axis it deserves. This is the right way to combine a relationship view with a distribution view: use subplots, not an overlay. The shape of the histogram (roughly bell-shaped, peaking around 3.0 cm) is now easy to read.

The key argument is the `row=`, `col=` pair on each `add_trace` call, telling Plotly **where** to place the trace in the grid.

## Adding interactivity

This is where Plotly really stands out. Two of the most useful patterns:

### Range slider

Lets the user zoom into a portion of the x-axis.

In [ ]:
fig = go.Figure(
    data=go.Histogram(x=df['sepal_width'], nbinsx=40, marker_color=color[0]),
    layout=go.Layout(
        title=dict(text='Sepal width, drag the slider to zoom', x=0.5),
        xaxis=go.layout.XAxis(
            title='Sepal width',
            rangeslider=go.layout.xaxis.Rangeslider(visible=True)
        )
    )
)

fig.show()

**Interpretation**

The histogram itself confirms what we saw before: sepal width is concentrated around 3.0 cm with a roughly symmetric, slightly skewed distribution. The real value of this plot, however, is the slider underneath. Drag its handles to zoom into a narrower range (say 2.5 to 3.5 cm) and the main chart updates to that window with finer bin resolution. This is one of Plotly's signature interactivity features, and it is especially useful for time-series data where users want to drill into a specific period without losing the global context.

### Dropdown menu, switch between visualizations

The pattern: add multiple traces (only the first visible), then attach buttons that toggle visibility.

In [ ]:
fig = go.Figure()

fig.add_trace(go.Histogram(x=df['sepal_width'], marker_color=color[0]))
fig.add_trace(go.Box(x=df['sepal_width'], marker_color=color[1], visible=False))
fig.add_trace(go.Violin(x=df['sepal_width'], line_color=color[2], visible=False))

fig.update_layout(
    title=dict(text='Sepal width, pick a visualization', x=0.5),
    showlegend=False,
    updatemenus=[go.layout.Updatemenu(
        active=0,
        buttons=[
            go.layout.updatemenu.Button(
                label='Histogram', method='update',
                args=[{'visible': [True, False, False]}]
            ),
            go.layout.updatemenu.Button(
                label='Box plot', method='update',
                args=[{'visible': [False, True, False]}]
            ),
            go.layout.updatemenu.Button(
                label='Violin plot', method='update',
                args=[{'visible': [False, False, True]}]
            ),
        ]
    )]
)

fig.show()

**Interpretation**

The same data, three different lenses. The histogram shows raw counts per bin, useful for spotting the modal value and any gaps. The box plot summarizes the distribution as median, quartiles, and outliers, useful for quick comparisons or for spotting extreme values. The violin plot combines a box plot with a smoothed density estimate (KDE), giving the best feel for the actual shape of the distribution including any bimodality. The dropdown pattern is powerful in dashboards where the user can choose the level of detail they want without you having to commit to one view.

## Maps

Plotly has two main map traces:

- `go.Scattergeo`: points at lat/lon coordinates (good for store networks, individual events)
- `go.Choropleth`: colored regions (good for sales by country/state)

### Choropleth

ISO 3-letter country codes match Plotly's built-in country boundaries automatically; no shapefiles to load.

In [ ]:
country_sales = pd.DataFrame({
    'country': ['France', 'Germany', 'Italy', 'Spain', 'United Kingdom'],
    'code':    ['FRA',    'DEU',     'ITA',   'ESP',   'GBR'],
    'sales':   [4500000, 6800000, 3200000, 2900000, 5600000]
})

fig = go.Figure(data=go.Choropleth(
    locations=country_sales['code'],
    z=country_sales['sales'],
    text=country_sales['country'],
    colorscale=custom_colorscale,
    colorbar=dict(title='Sales ($)')
))

fig.update_layout(
    title=dict(text='European sales distribution', x=0.5),
    geo=dict(scope='europe', projection_type='natural earth')
)

fig.show()

**Interpretation**

Germany is the strongest market by a clear margin (\$6.8M), followed by the UK (\$5.6M) and France (\$4.5M). Italy and Spain are notably behind at around \$3M each. The choropleth makes the geographic pattern immediate: the big northern markets (Germany, UK) dominate, while southern Europe (Italy, Spain) underperforms in this dataset. The custom colorscale runs from deep navy (low sales) up through blue, teal, orange, and pink (high sales), so the eye is drawn straight to Germany. 

> For any sales review the next question is obvious: why are Italy and Spain lagging, and what would it take to close the gap?

### Scattergeo

This trace plots points at **lat/lon coordinates**, with the same styling options as a regular scatter. It's ideal for store locations, event occurrences, or any point data with geographic coordinates. You can customize the map projection and scope to focus on specific regions.

Here an example of store locations across the US, colored by sales volume. 

In [ ]:
fig = go.Figure(go.Scattergeo(
    lon=[-74.0060, -118.2437, -87.6298],  # New York, Los Angeles, Chicago
    lat=[40.7128, 34.0522, 41.8781],
    text=['NY Store', 'LA Store', 'Chicago Store'],
    marker=dict(
        size=[20, 30, 15],
        color=[color[0], color[1], color[2]],
        line=dict(width=1, color='black')
    )
))
fig.update_layout(
    title='Store Locations and Sales Volume',
    geo=dict(
        scope='usa',
        projection_type='albers usa',
        showland=True,      
        landcolor='lightgray',
        subunitcolor='white'
    )
)   
fig.show()

## Resources 📚📚

- [Plotly Python homepage](https://plotly.com/python/) 
- [Full API reference](https://plotly.com/python-api-reference/) 
- [Creating and updating figures](https://plotly.com/python/creating-and-updating-figures/) 
- [Subplots](https://plotly.com/python/subplots/)
- [Range sliders](https://plotly.com/python/range-slider/)
- [Custom buttons & dropdowns](https://plotly.com/python/custom-buttons/)
- [Animations](https://plotly.com/python/animations/) 
- [Hover text customization](https://plotly.com/python/hover-text-and-formatting/)
- [Geographic maps overview](https://plotly.com/python/maps/)
- [Choropleth maps](https://plotly.com/python/choropleth-maps/)
- [Scatter on maps](https://plotly.com/python/scatter-plots-on-maps/)
- [Mapbox-based maps](https://plotly.com/python/scattermapbox/) 
- [Themes & templates](https://plotly.com/python/templates/) 
- [Built-in colorscales](https://plotly.com/python/builtin-colorscales/)